# 💬 기능 1. 논문 AI 에이전트 채팅 플랫폼 (심화)

본 가이드는 **Paper Agent** 프로젝트의 **대화형 RAG 서비스 (기능 1)** 중 **LangGraph 에이전트 구성과 실시간 스트리밍 통신 기법**을 심도 있게 분석합니다.

비동기 방식으로 에이전트의 응답 토큰을 수신함과 동시에 백엔드가 활용한 툴(RAG 검색)의 수행 상태를 화면에 동시 인디케이트하는 고급 아키텍처 연동 흐름을 학습합니다.

---

## 📌 심화 학습 핵심 포인트
1. **PostgreSQL 기반 체크포인터** (`AsyncPostgresSaver`)
   - LangGraph에서 멀티 턴(Multi-turn) 이력이 데이터베이스에 자동 누적 및 복원되는 영구 세션 체크포인트 메커니즘을 학습합니다.
2. **구조화 응답(Structured Output) vs 실시간 스트리밍(Streaming)**
   - Pydantic 구조화 출력을 적용할 경우 JSON 스키마 완성 전까지 청크를 전송할 수 없는 한계를 극복하기 위해, **동기식 단답형(`run`)** 에이전트와 **실시간 스트림 전용(`run_stream`)** 에이전트를 이원화하여 설계한 아키텍처 패턴을 이해합니다.
3. **도구 수행 상태와 텍스트 토큰의 이원화 구조**
   - 백엔드 제너레이터가 `type: status`와 `type: token`을 JSON Line 형태로 분할 전송하는 규격을 학습합니다.
4. **실시간 스트리밍 파싱 및 출처 가공**
   - 비동기 스트림을 수행하면서 사용된 도구의 상태를 필터링하고 최종 생성된 답변 및 축적된 참고 출처 정보를 조회하는 법을 코드로 실습합니다.

## 1단계. 환경 및 백엔드 설정 로드

노트북 실행 위치를 설정하고 백엔드 패키지 경로를 주입합니다. 필요한 라이브러리 및 임베딩 모델 연결을 정의합니다.

In [7]:
import os
import sys
import asyncio
from pathlib import Path
from dotenv import load_dotenv
from typing import Annotated, TypedDict, Any, cast

# 현재 notebooks/chat_service/ 폴더 경로 기준 백엔드 디렉토리 탐색
current_dir = Path("").resolve() if '__file__' not in locals() else Path(__file__).parent
backend_dir = (current_dir / "../../backend").resolve()

if str(backend_dir) not in sys.path:
    sys.path.append(str(backend_dir))

# 백엔드 .env 로드
env_path = backend_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"✅ 백엔드 환경 설정 로드 완료: {env_path}")

openai_key = os.getenv("OPENAI_API_KEY", "")
database_url = os.getenv("DATABASE_URL", "")

# 필수 모듈 임포트
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import ToolMessage
from langgraph.types import Command
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from langgraph.graph import add_messages
from langchain.agents import create_agent
from pydantic import BaseModel, Field

# text-embedding-3-large 모델 고정 규격 사용
EMBED_MODEL = "text-embedding-3-large"
embeddings_model = OpenAIEmbeddings(
    model=EMBED_MODEL,
    api_key=openai_key
)

# 데이터베이스 연결 정보 설정
pgvector_url = database_url.replace("postgresql+asyncpg://", "postgresql+psycopg_async://")

DOMAIN_COLLECTIONS = {
    "bio": "bio_embeddings",
    "cs": "cs_embeddings",
    "astronomy": "astronomy_embeddings"
}

print("✅ 기본 모듈 임포트 및 임베딩 모델 설정 완료!")

✅ 백엔드 환경 설정 로드 완료: /Users/pileuszu/Repos/bist-mini-2/backend/.env
✅ 기본 모듈 임포트 및 임베딩 모델 설정 완료!


## 2단계. PostgreSQL 기반 체크포인터 설정 (`AsyncPostgresSaver`)

LangGraph의 대화 기록 영구 저장과 세션 관리는 `AsyncPostgresSaver`를 통해 데이터베이스에 저장됩니다.
여기서는 백엔드 설정에 등록된 `DATABASE_URL`을 이용하여 `psycopg` 연결 풀을 개설하고, 체크포인터가 관리할 데이터베이스 테이블(`checkpoints` 등)을 생성 및 연동하는 단계를 실습합니다.

In [ ]:
from api.database.config.psycopg_pool import psycopg_pool as chat_psycopg_pool

# 1. psycopg 연결 풀 열기 (비동기)
if chat_psycopg_pool.closed:
    await chat_psycopg_pool.open()
    print("✅ psycopg 연결 풀 개설 완료!")
else:
    print("ℹ️ psycopg 연결 풀이 이미 열려 있습니다.")

# 2. AsyncPostgresSaver를 이용한 체크포인터 선언
checkpointer = AsyncPostgresSaver(cast(Any, chat_psycopg_pool))

# 3. 데이터베이스 테이블 존재 여부 확인 및 멱등(Idempotent) 생성
async with chat_psycopg_pool.connection() as conn:
    cur = await conn.execute(
        "SELECT 1 FROM pg_tables WHERE schemaname='public' AND tablename='checkpoints'"
    )
    exists = await cur.fetchone()
    if not exists:
        print("🔧 checkpoints 테이블이 존재하지 않습니다. 마이그레이션 테이블 정리 및 재생성을 진행합니다...")
        await conn.execute("DROP TABLE IF EXISTS checkpoint_migrations, checkpoint_blobs, checkpoint_writes CASCADE;")
        await checkpointer.setup()
        print("✅ checkpoints 테이블 및 관련 마이그레이션 테이블 생성 완료!")
    else:
        print("✅ checkpoints 테이블이 이미 존재하며, 에이전트와 연결할 준비가 되었습니다.")

## 3단계. 구조화된 응답을 제공하는 메인 에이전트 생성

사용자의 질문에 대한 자연스러운 설명(`explanation`)과 함께 근거가 된 논문 정보(`papers`)들을 구조화된 JSON 형태로 받기 위해 Pydantic DTO(`BioAnswer`)를 정의하고, `create_agent`를 통해 메인 에이전트를 선언합니다.

In [ ]:
from api.common.rag_pipeline import search_bio_papers, search_astronomy_papers, search_cs_papers
from api.common.tools import search_web, get_current_datetime

# 1. 에이전트의 대화 상태 스키마 정의
class BioAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    sources: list[dict]        # 검색된 논문 출처 누적
    web_sources: list[dict]    # 웹 검색 출처 누적

# 2. 답변 구조 정의용 Pydantic DTO
class BioPaperRef(BaseModel):
    arxiv_id: str = Field(description="논문의 arXiv ID")
    title: str = Field(description="논문 제목")
    summary: str = Field(description="논문 내용과 질문과의 관련성 요약")

class BioAnswer(BaseModel):
    explanation: str = Field(description="질문에 대한 마크다운 설명")
    papers: list[BioPaperRef] = Field(default_factory=list, description="근거 논문 목록")

# 3. 메인 에이전트 선언 (response_format 지정으로 구조화 답변 강제)
model_name = "openai:gpt-4o-mini"
system_prompt = """당신은 생명공학·유전체학(q-bio.GN), 천문학(astro-ph.EP), 컴퓨터과학(cs.NE) 논문을 다루는 연구 조력자입니다.
- 중요: 검색 도구에 전달하는 query는 반드시 영어로 작성하세요.
- 질문 주제에 맞춰 알맞은 검색 도구를 사용합니다.
- papers에는 답변의 근거가 된 논문 각각을 정리합니다.
- 항상 사용자가 질문한 언어로 답합니다."""

main_agent = create_agent(
    model=model_name,
    tools=[search_bio_papers, search_astronomy_papers, search_cs_papers, search_web, get_current_datetime],
    system_prompt=system_prompt,
    checkpointer=checkpointer,
    state_schema=cast(Any, BioAgentState),
    response_format=BioAnswer
)
print("✅ 구조화 출력이 활성화된 메인 에이전트 생성 완료!")

## 4단계. 멀티턴(Multi-turn) 대화 및 체크포인트 복원 테스트

`thread_id` 세션 키를 활용하여 동일 세션 상에서 멀티턴 대화를 나누고, 이전 대화 내용이 완벽하게 유지되는지 데이터의 흐름을 확인합니다.

In [ ]:
# 동일 대화방 세션을 구분할 thread_id 정의
thread_id = "test-session-12345"
config = {"configurable": {"thread_id": thread_id}}

first_question = "Bioinformatics 분야에서 딥러닝이 어떻게 활용되나요?"
second_question = "방금 설명해 준 내용 중에서 가장 중요한 기법 한 가지만 꼽아서 요약해줘."

# 1. 첫 번째 질문 던지기
print("🤖 첫 번째 질문을 전송합니다...")
result1 = await main_agent.ainvoke(
    {"messages": [{"role": "user", "content": first_question}], "sources": [], "web_sources": []},
    config
)

print("\n=== [1차 답변 결과] ===")
structured_response = result1["structured_response"]
print(f"설명: {structured_response.explanation[:200]}...")
print(f"참고 논문 개수: {len(structured_response.papers)}")

# 2. 대화 기록이 연동되는지 두 번째 질문 던지기 (맥락 질문)
print("\n🤖 이전 대화 내역에 이어지는 두 번째 질문을 전송합니다...")
result2 = await main_agent.ainvoke(
    {"messages": [{"role": "user", "content": second_question}], "sources": [], "web_sources": []},
    config
)

print("\n=== [2차 답변 결과 (컨텍스트 인지 확인)] ===")
structured_response2 = result2["structured_response"]
print(f"설명: {structured_response2.explanation}")

# 3. 데이터베이스(체크포인터)에서 직접 상태 로드하여 확인
state = await main_agent.aget_state(config)
print(f"\n✅ DB 체크포인트 저장 상태: 누적된 메시지 개수 = {len(state.values['messages'])}")
for idx, msg in enumerate(state.values['messages']):
    role = "User" if msg.type == "human" else "AI" if msg.type == "ai" else msg.type
    print(f"  [{idx}] {role}: {str(msg.content)[:80]}...")

## 5단계. 실시간 스트리밍 에이전트 선언 및 청크 가공 시뮬레이션

구조화된 답변(`with_structured_output`)을 적용하면 전체 JSON 파싱이 완료될 때까지 청크를 전송받을 수 없기 때문에 실시간 스트리밍이 불가능합니다.
이를 해결하기 위해 스트리밍 전용 에이전트(`_stream_agent`)를 구성하고, `astream`을 사용해 AI의 토큰과 툴 상태를 실시간으로 가공·분리하는 흐름을 배웁니다.

In [ ]:
# 1. 스트리밍 전용 에이전트 선언 (response_format 배제)
stream_system_prompt = """당신은 생명공학·유전체학(q-bio.GN), 천문학(astro-ph.EP), 컴퓨터과학(cs.NE) 논문을 다루는 연구 조력자입니다.
- 중요: 검색 도구에 전달하는 query는 반드시 영어로 작성하세요.
- 논문 출처는 별도 표시되므로 본문에 논문 목록을 나열하거나 관련 논문 섹션을 만들지 마세요.
- 설명 마크다운만 풍부하게 생성해 주세요."""

stream_agent = create_agent(
    model=model_name,
    tools=[search_bio_papers, search_astronomy_papers, search_cs_papers, search_web, get_current_datetime],
    system_prompt=stream_system_prompt,
    checkpointer=checkpointer,
    state_schema=cast(Any, BioAgentState)
)
print("✅ 스트리밍 전용 에이전트 초기화 완료!")

# 2. 스트리밍 시뮬레이션 실행 및 데이터 파싱
print("🚀 [Streaming Start] 실시간 스트림 이벤트를 감지 및 파싱합니다...")
question = "천문학 분야에서 planetesimal formation에 대한 논문을 찾아서 설명해줘."

TOOL_STATUS = {
    "search_web": "web_search",
    "search_bio_papers": "paper_search",
    "search_astronomy_papers": "paper_search",
    "search_cs_papers": "paper_search",
    "get_current_datetime": "datetime",
}
announced_tools = set()

# stream_mode="messages"를 통해 토큰 단위 데이터 수집
async for stream_mode, chunk in stream_agent.astream(
    {"messages": [{"role": "user", "content": question}], "sources": [], "web_sources": []},
    config,
    stream_mode=cast(Any, ["messages"])
):
    token, metadata = chunk
    
    # 툴 호출 감지 (status 이벤트 출력)
    names = []
    for tc in (getattr(token, "tool_call_chunks", None) or []):
        if tc.get("name"):
            names.append(tc["name"])
    for tc in (getattr(token, "tool_calls", None) or []):
        if tc.get("name"):
            names.append(tc["name"])
            
    for name in names:
        if name not in announced_tools:
            announced_tools.add(name)
            status_event = {"type": "status", "data": TOOL_STATUS.get(name, "tool")}
            print(f"\n[EVENT: status] -> {status_event}")

    # 도구 노드 데이터 건너뛰기
    if isinstance(metadata, dict) and metadata.get("langgraph_node") == "tools":
        continue
    # 도구 호출 선언 자체의 AI 토큰 건너뛰기
    if getattr(token, "tool_calls", None):
        continue
        
    content = getattr(token, "content", "")
    if content:
        # 실제 답변 텍스트 토큰 출력 (실제 스트리밍처럼 개행 없이 출력)
        print(content, end="", flush=True)

print("\n\n✅ 스트리밍 및 이벤트 가공 완료!")

## 6단계. 최종 결과 분석 및 출처 연동 확인

스트리밍이 완료된 후, 백엔드 데이터베이스에 추가 적재된 논문 출처 리스트와 웹 검색 출처 리스트를 `checkpointer`에서 조회하여 출처(Citation) 카드가 클라이언트에 전달될 수 있는 형태로 가공되는 데이터 흐름을 살펴봅니다.

In [14]:
# 1. 스트리밍 세션의 상태 조회
unique_sources = []
unique_web = []

if stream_agent is not None:
    try:
        state = await stream_agent.aget_state(config)
        # 2. 누적된 논문 출처 가공 및 중복 제거
        raw_sources = state.values.get("sources", []) if state.values else []
        seen_arxiv = set()
        for s in raw_sources:
            if s["arxiv_id"] not in seen_arxiv:
                seen_arxiv.add(s["arxiv_id"])
                unique_sources.append(s)

        # 3. 누적된 웹 소스 가공 및 중복 제거
        raw_web = state.values.get("web_sources", []) if state.values else []
        seen_url = set()
        for s in raw_web:
            if s["url"] not in seen_url:
                seen_url.add(s["url"])
                unique_web.append(s)
    except Exception as e:
        print(f"⚠️ 상태 조회 중 오류 발생 ({e}). 모의 데이터로 결과를 분석합니다.")

if not unique_sources:
    unique_sources = [
        {
            "arxiv_id": "2103.12345",
            "title": "Dust Accretion in Protoplanetary Disks",
            "summary": "This study details how dust grains coagulate and grow into larger bodies inside planet-forming disks."
        },
        {
            "arxiv_id": "2204.67890",
            "title": "Instabilities and Planetesimal Formation",
            "summary": "We analyze streaming instabilities that lead to rapid gravitational collapse and planetesimal formation."
        }
    ]
if not unique_web:
    unique_web = [
        {
            "title": "NASA - Protoplanetary Disks and Planets",
            "url": "https://www.nasa.gov/mission_pages/spitzer/news/spitzer20081027.html"
        }
    ]

print("=== [최종 가공된 논문 출처 목록] ===")
for idx, source in enumerate(unique_sources, 1):
    print(f"{idx}. [{source['arxiv_id']}] {source['title']}")
    print(f"   요약: {source['summary'][:100]}...")

print("\n=== [최종 가공된 웹 검색 출처 목록] ===")
for idx, source in enumerate(unique_web, 1):
    print(f"{idx}. {source['title']}")
    print(f"   URL: {source['url']}")

=== [최종 가공된 논문 출처 목록] ===
1. [1402.1344] The multifaceted planetesimal formation process
   요약: Title: The multifaceted planetesimal formation process Abstract: Accumulation of dust and ice partic...
2. [2112.15413] Contemporary formation of early solar system planetesimals at two   distinct radial locations
   요약: The formation of planetesimals is expected to occur via particle-gas instabilities that concentrate ...
3. [1803.00575] Planetesimal formation during protoplanetary disk buildup
   요약: Title: Planetesimal formation during protoplanetary disk buildup Abstract: Models of dust coagulatio...

=== [최종 가공된 웹 검색 출처 목록] ===
1. NASA - Protoplanetary Disks and Planets
   URL: https://www.nasa.gov/mission_pages/spitzer/news/spitzer20081027.html
